# Part 2: 다양한 모델 포맷 - 실습

- 실습 1: PyTorch 모델 저장 방식 이해
- 실습 2: ONNX 변환과 검증
- 실습 3: ONNX Runtime으로 추론하기

In [ ]:
# ======================================================================================
# [실습 목표] Part 2: 모델 포맷팅과 ONNX(Open Neural Network Exchange) 실습
# ======================================================================================
#
# 1. [Why] 왜 이 짓을 하는가? (필요성)
#    - PyTorch로 만든 모델은 PyTorch가 깔린 파이썬 환경에서만 돌아갑니다.
#    - 하지만 실제 서비스는 C++ 서버, 아이폰(iOS), 안드로이드, 웹브라우저 등 다양한 곳에서 돌아가야 합니다.
#    - 그래서 "어디서든 돌아가는 만능 포맷"인 ONNX로 변환하는 기술이 필수적입니다.
#
# 2. [Goal] 무엇을 확인하는가? (목표)
#    - 모델을 저장할 때 껍데기(코드)와 알맹이(가중치)를 분리해서 저장하는 법을 배웁니다.
#    - PyTorch 모델(.pth)을 ONNX 모델(.onnx)로 변환하고, 구조가 깨지지 않았는지 검증합니다.
#    - ONNX Runtime을 돌려서 PyTorch보다 속도가 얼마나 빨라지는지 직접 잽니다.
#
# 3. [Key Tech] 핵심 기술 용어
#    - state_dict: 모델의 모든 가중치(Weight)와 편향(Bias)이 담긴 딕셔너리 파일.
#    - Serialization(직렬화): 메모리에 떠 있는 객체(모델)를 파일로 저장 가능한 형태로 바꾸는 것.
#    - ONNX: AI 모델의 "PDF" 같은 표준 포맷. (TensorFlow에서 만들든 PyTorch에서 만들든 다 호환됨)
#    - ONNX Runtime (ORT): ONNX 파일을 읽어서 초고속으로 실행시켜주는 전용 엔진.
#
# ======================================================================================

# [기계적 구동]
# 필요한 라이브러리들을 설치합니다.
# - onnx: ONNX 포맷을 다루는 기본 도구
# - onnxruntime: ONNX 모델을 실제로 실행(Inference)하는 고성능 엔진
!pip install onnx onnxruntime onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.5 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os

print(f"PyTorch 버전: {torch.__version__}")

PyTorch 버전: 2.9.0+cu126


---
## 실습 1: PyTorch 모델 저장 방식 이해

### 학습 목표
- state_dict vs 전체 모델 저장의 차이 이해
- 모델 저장/로드 방법 익히기
- 저장된 파일 크기 비교

### 1-1. 간단한 CNN 모델 정의

In [ ]:
# [설명] 손글씨(MNIST) 같은 이미지를 분류하는 전형적인 CNN(Convolutional Neural Network) 구조입니다.
class SimpleCNN(nn.Module):
    """MNIST 분류를 위한 간단한 CNN 모델"""
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1채널(흑백) 입력 -> 32개 특징맵 출력, 3x3 필터 사용
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2) # 이미지 크기 반토막 내기 (압축)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128) # 완전 연결 계층 (분류기)
        self.fc2 = nn.Linear(128, 10) # 0~9까지 10개 숫자 분류
        self.relu = nn.ReLU()

    def forward(self, x):
        # [데이터 흐름]
        # 입력(28x28) -> conv1 -> relu -> pool -> (14x14, 32채널)
        # -> conv2 -> relu -> pool -> (7x7, 64채널)
        # -> 평탄화(view) -> 은닉층(fc1) -> 출력층(fc2)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7) # 3차원 큐브 모양 데이터를 1줄로 핌
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 모델 생성 (메모리에 인스턴스화)
model = SimpleCNN()
print(model)

# 파라미터 수 확인
# [기계적 구동] 모델 내 모든 텐서를 순회하며 원소 개수(numel)를 합산합니다.
total_params = sum(p.numel() for p in model.parameters())
print(f"\n총 파라미터 수: {total_params:,}개")

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)

총 파라미터 수: 421,642개


### 1-2. 모델 저장 방법 비교
---

✅ 1) **state_dict 저장 (추천)**

  - 가중치(weight)만 저장
  - 모델 구조 코드는 따로 필요
  - 장점: 유연성, 용량 작음, 호환성 좋음
  - 단점: 모델 클래스 정의 필요

```
torch.save(model.state_dict(), "model.pth")

model = MyModel()
model.load_state_dict(torch.load("model.pth"))
model.eval()
```


⚠️ 2) 전체 모델 저장 (비추천)

  - 모델 구조 + 가중치 통째로 저장
  - 로드시 저장할 때의 디렉토리 구조나 클래스 정의가 불러올 때와 완벽하게 일치해야 함.
  - 장점: 로드가 간단
  - 단점: 코드/클래스 경로 바뀌면 로드 실패 가능
```
torch.save(model, "full_model.pth")

model = torch.load("full_model.pth")
model.eval()
```


In [ ]:
# 저장 디렉토리 생성
os.makedirs('saved_models', exist_ok=True)

# 방법 1: state_dict만 저장 (권장)
# [기계적 구동] 모델의 파라미터 맵(Dictionary)만 추출하여 직렬화(Pickle) 후 디스크에 씁니다.
torch.save(model.state_dict(), 'saved_models/model_state_dict.pth')

# 방법 2: 전체 모델 저장 (pickle 사용)
# [기계적 구동] 파이썬 객체 자체를 통째로 얼려서 저장합니다.
# 여기에는 파라미터뿐만 아니라 디렉토리 경로, 클래스 정의 정보까지 포함되어 나중에 경로가 바뀌면 에러가 납니다.
torch.save(model, 'saved_models/model_full.pth')

# 파일 크기 비교
size_state_dict = os.path.getsize('saved_models/model_state_dict.pth')
size_full = os.path.getsize('saved_models/model_full.pth')

print("모델 저장 방식 비교")
print("=" * 50)
print(f"state_dict 저장: {size_state_dict:,} bytes ({size_state_dict/1024:.2f} KB)")
print(f"전체 모델 저장:  {size_full:,} bytes ({size_full/1024:.2f} KB)")

모델 저장 방식 비교
state_dict 저장: 1,690,251 bytes (1650.64 KB)
전체 모델 저장:  1,692,151 bytes (1652.49 KB)


### 1-3. 저장된 모델 불러오기

In [ ]:
# 방법 1: state_dict 불러오기
# [주의] 껍데기(SimpleCNN 클래스)는 우리가 코드로 미리 만들어놔야 합니다.
model_loaded = SimpleCNN()    # 빈 깡통 모델 생성 (가중치는 랜덤 초기화 상태)
state_dict = torch.load(
    "saved_models/model_state_dict.pth",
    map_location="cpu", # GPU에서 저장했어도 강제로 CPU로 불러옵니다.
    weights_only=True,  # 보안 옵션: 이상한 코드가 실행되지 않게 가중치만 딱 로드함.
)
# [기계적 구동] 로드한 딕셔너리의 키(예: 'conv1.weight')와 모델의 키를 맞춰서 값을 덮어씁니다.
model_loaded.load_state_dict(state_dict)
model_loaded.eval()           # 평가 모드로 설정 (Dropout, BatchNorm 등을 고정)
print("state_dict 로드 완료!")


# 방법 2: 전체 모델 불러오기
# [기계적 구동] 파일 안에 클래스 정보가 다 있어서 `SimpleCNN()` 호출 없이 바로 객체가 튀어나옵니다.
model_full_loaded = torch.load(
    "saved_models/model_full.pth",
    map_location="cpu",
    weights_only=False, # 전체 객체를 로드해야 하므로 False (보안상 위험할 수 있음)
)
model_full_loaded.eval()
print("전체 모델 로드 완료!")

# 두 모델이 같은 결과를 출력하는지 확인
dummy_input = torch.randn(1, 1, 28, 28)
with torch.no_grad():
    output1 = model_loaded(dummy_input)
    output2 = model_full_loaded(dummy_input)

# [검증] torch.allclose: 두 텐서의 모든 값이 오차 범위 내에서 같은지 확인
print(f"\n---- 두 모델의 출력이 동일한가? ----: {torch.allclose(output1, output2)}")

state_dict 로드 완료!
전체 모델 로드 완료!

---- 두 모델의 출력이 동일한가? ----: True


### 🎯 실습 1-1: state_dict 내용 확인하기

state_dict에 저장된 가중치의 구조와 shape을 확인해보기
1) state_dict
    - 모델이 학습한 파라미터(가중치/편향)들의 딕셔너리
    - key는 레이어 이름, value는 텐서(weight/bias)
    - 예: fc1.weight, fc1.bias

2) key, value
    - key : 파라미터 이름 (어떤 레이어인지)
    - value : 그 파라미터 값(텐서)
    - value.shape : 파라미터 행렬의 크기 (예: [128, 784])
    - dtype : 저장 형식 (보통 float32)
3) numel()
    - value.numel() = 텐서 안에 숫자가 총 몇 개 들어있는지
    - 이걸 다 합친 것이 total_elements

In [ ]:
state_dict = model.state_dict()

print("모델이 저장하는 값(state_dict) 확인하기")
print("=" * 70)

total_elements = 0
for key, value in state_dict.items():
    num_elements = value.numel()
    total_elements += num_elements
    # 각 레이어별로 가중치가 어떻게 생겼는지(Shape) 눈으로 확인합니다.
    print(f"{key:25} | shape: {str(value.shape):20} | dtype: {value.dtype} | elements: {num_elements:,}")

print("=" * 70)
print(f"총 파라미터 개수(가중치 숫자 개수): {total_elements:,}")
# [계산] 파라미터 개수 * 4바이트(FP32) = 필요한 메모리 용량
print(f"FP32로 저장했을 때 필요한 메모리(용량): {total_elements * 4 / 1024:.2f} KB")

state_dict 내용 분석
conv1.weight              | shape: torch.Size([32, 1, 3, 3]) | dtype: torch.float32 | elements: 288
conv1.bias                | shape: torch.Size([32])     | dtype: torch.float32 | elements: 32
conv2.weight              | shape: torch.Size([64, 32, 3, 3]) | dtype: torch.float32 | elements: 18,432
conv2.bias                | shape: torch.Size([64])     | dtype: torch.float32 | elements: 64
fc1.weight                | shape: torch.Size([128, 3136]) | dtype: torch.float32 | elements: 401,408
fc1.bias                  | shape: torch.Size([128])    | dtype: torch.float32 | elements: 128
fc2.weight                | shape: torch.Size([10, 128]) | dtype: torch.float32 | elements: 1,280
fc2.bias                  | shape: torch.Size([10])     | dtype: torch.float32 | elements: 10
총 파라미터 개수(가중치 숫자 개수): 421,642
FP32로 저장했을 때 필요한 메모리(용량): 1647.04 KB


---
## 실습 2: ONNX 변환과 검증

### 학습 목표
- PyTorch 모델을 ONNX로 변환하는 방법 익히기
- ONNX 모델 구조 확인
- 변환 전후 결과 검증

### 2-1. PyTorch → ONNX 변환

✅ **ONNX란?**

ONNX는 PyTorch 모델을 다른 환경(ONNX Runtime, TensorRT 등)에서도 실행할 수 있게 변환하는 표준 포맷이에요.

✅ **torch.onnx.export() 주요 파라미터 설명**
- model : 변환할 PyTorch 모델
- dummy_input : 모델 입력 예시(입력 shape 결정)
- onnx_path : 저장할 파일 경로
- export_params=True : 학습된 가중치(weight) 포함
- opset_version=18 : ONNX 연산 규칙 버전(표준 버전)
- do_constant_folding=True : 고정 계산은 미리 계산해 최적화
- input_names / output_names : 입력/출력 이름 지정
- dynamic_axes : batch size를 가변으로 설정

In [ ]:
import onnx

# [아주 중요] 모델을 평가 모드로 설정
model.eval()

# 더미 입력 생성 (MNIST: 1채널, 28x28)
# [기계적 구동] 이 더미 데이터를 모델에 한 번 흘려보냅니다(Tracing).
# --------------------------------------------------------------------------------------
# ✅ "데이터 경로를 추적해 지도를 그린다"는 말의 의미 (Tracing):
# 1. [예행연습] dummy_input을 모델에 통과시켜 실제 데이터 흐름을 관찰합니다.
# 2. [지도 작성] 어떤 연산(Conv, ReLU 등)이 어떤 순서로 일어나는지 기록하여 '그래프'를 만듭니다.
# 3. [독립성] 이렇게 만든 '지도(그래프)'가 있으면 파이썬 없이도 모델을 실행할 수 있게 됩니다.
# --------------------------------------------------------------------------------------
dummy_input = torch.randn(1, 1, 28, 28)

# ONNX로 내보내기
onnx_path = 'saved_models/model.onnx'

torch.onnx.export(
    model,                  # 변환할 모델
    dummy_input,            # 모델 입력 예시 (Tracing용)
    onnx_path,              # 저장 경로
    export_params=True,     # 가중치를 파일 안에 같이 저장 (False면 구조만 저장)
    opset_version=18,       # ONNX 버전 (최신 버전 사용 권장)
    do_constant_folding=True, # 상수 폴딩: 미리 계산 가능한 상수(예: 3+5)는 8로 저장해서 속도 최적화
    input_names=['input'],  # 나중에 다른 툴에서 입력 찾을 때 쓸 이름
    output_names=['output'],# 나중에 다른 툴에서 출력 찾을 때 쓸 이름
    dynamic_axes={          # [중요] 배치 크기(0번 축)는 고정하지 말고 가변적으로 둬라
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"ONNX 모델 저장 완료: {onnx_path}")
print(f"파일 크기: {os.path.getsize(onnx_path) / 1024:.2f} KB")

/tmp/ipython-input-2339855264.py:12: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX 모델 저장 완료: saved_models/model.onnx
파일 크기: 11.75 KB


### 2-2. ONNX 모델 검증

In [ ]:
# ======================================================================================
# ✅ [상세 설명] ONNX 모델 로드 및 구조 설계도 검증
# ======================================================================================
# 1. onnx.load(): 
#    - ONNX 파일은 'Protobuf(Protocol Buffers)'라는 구글이 만든 이진 형식을 씁니다.
#    - 아주 압축된 0과 1의 데이터 덩어리를 다시 파이썬이 읽을 수 있는 '모델 객체'로 복구합니다.
#
# 2. onnx.checker.check_model(): 
#    - 단순히 파일이 잘 열리는지 보는 게 아니라, 내부 '설계도(Graph)'를 전수 조사합니다.
#    - [연결성 확인] "A 레이어의 출력이 10개인데, B 레이어가 입력으로 5개만 받진 않는가?" (끊긴 구간 체크)
#    - [표준 준수] "ONNX 표준에 없는 이상한 듣보잡 연산자를 쓰고 있진 않은가?"
#    - [속성 확인] 각 연산(Conv 등)에 필요한 필수 정보(커널 사이즈, 스트라이드 등)가 다 들어있는가?
#
# 3. 왜 하는가? (필요성):
#    - 변환 과정에서 미세한 에러가 나면, 파일은 생성되지만 막상 실행하면 서버가 죽어버릴 수 있습니다.
#    - 추론 세션(InferenceSession)을 만들기 전, 이 검증을 통과해야 안심하고 배포할 수 있습니다.
# ======================================================================================

# ONNX 모델 로드
onnx_model = onnx.load(onnx_path)

# 모델 유효성 정밀 검사
try:
    # 설계도에 논리적인 결함이 있으면 여기서 에러가 발생합니다.
    onnx.checker.check_model(onnx_model)
    print("✅ [검증 통과] ONNX 모델의 계산 그래프가 논리적으로 완벽합니다!")
except Exception as e:
    print(f"❌ [검증 실패] 모델 구조에 문제가 발견되었습니다: {e}")

✅ ONNX 모델 검증 성공!


### 2-3. ONNX 모델 구조 확인

In [ ]:
# ======================================================================================
# ✅ [상세 설명] 모델의 '사용 설명서(Metadata)' 뜯어보기
# 1. IR 버전: ONNX라는 표준 언어의 "문법 버전"입니다. 숫자가 높을수록 최신 기능을 지원합니다.
# 2. Producer: 이 모델을 만든 '엔진'(PyTorch 등)이 무엇인지 확인하는 '족보' 정보입니다.
# 3. 입력/출력 정보: 모델의 '입구'와 '출구' 스펙을 확인합니다. 
#    - 어떤 이름으로 데이터를 던져줘야 하는지(input), 
#    - 결과값의 크기(Shape)는 어떠한지(예: 28x28 이미지, 10개의 숫자 분류)를 체크합니다.
# ======================================================================================

# 모델 메타데이터
print("ONNX 모델 정보")
print("=" * 50)
print(f"IR 버전: {onnx_model.ir_version}") # Intermediate Representation 버전
print(f"Producer: {onnx_model.producer_name}") # 누가 만들었니? (pytorch 2.x 등)
print(f"Model 버전: {onnx_model.model_version}")

# 입력 정보
# [기계적 구동] 그래프의 시작 노드(Input) 정보를 뜯어봅니다.
print("\n입력 정보:")
for input in onnx_model.graph.input:
    print(f"  이름: {input.name}")
    tensor_type = input.type.tensor_type
    # 차원 정보 추출 (배치 사이즈는 'batch_size'라는 문자열로 되어 있을 수 있음)
    dims = [d.dim_value if d.dim_value else d.dim_param for d in tensor_type.shape.dim]
    print(f"  Shape: {dims}")

# 출력 정보
print("\n출력 정보:")
for output in onnx_model.graph.output:
    print(f"  이름: {output.name}")

ONNX 모델 정보
IR 버전: 10
Producer: pytorch
Model 버전: 0

입력 정보:
  이름: input
  Shape: ['s77', 1, 28, 28]

출력 정보:
  이름: output


# ONNX 모델 메타 정보 정리

## 1. 모델 기본 정보 (Header)
- **IR 버전: 10**  
  ONNX 모델을 만드는 '문법 표준' 버전입니다. 숫자가 높을수록 최신 기능을 지원하며, 여기서는 10번 규칙으로 작성되었다는 뜻입니다.
- **Producer: pytorch**  
  이 모델의 태생(고향)입니다. PyTorch에서 `torch.onnx.export`를 통해 만들어졌음을 보여줍니다.
- **Model 버전: 0**  
  개발자가 모델에 수동으로 부여하는 버전 번호입니다. 따로 설정하지 않아 기본값인 0으로 표시되었습니다.

---

## 2. 입력 정보 (Input)
- **이름: input**  
  데이터를 넣을 때 사용해야 하는 변수 이름입니다. ONNX 런타임에서 실행할 때 `{'input': 데이터}` 형태로 이름을 맞춰줘야 합니다.
- **Shape: ['s77', 1, 28, 28]**  
  모델이 받는 데이터의 형상입니다.
  - `'s77'`: Batch Size(데이터 개수)가 올 자리인데, 숫자가 아닌 문자로 표시된 것은 **가변(Dynamic) 상태**라는 뜻입니다.  
    → 내보낼 때 `dynamic_axes` 설정을 했기 때문에 1장이든 10장이든 유연하게 받을 수 있습니다.
  - `1`: 채널 수 (흑백 이미지 1채널)
  - `28, 28`: 이미지의 가로, 세로 크기 (28x28 픽셀)

---

## 3. 출력 정보 (Output)
- **이름: output**  
  모델이 계산을 마치고 결과값을 내놓을 때 붙여주는 결과값의 이름입니다.

---

## 📌 요약
이 모델은 **PyTorch로 만든 28x28 흑백 이미지 분류 모델**이며,  
한 번에 몇 장의 데이터가 들어오든(`s77`) 상관없이 처리할 수 있는 **유연한 입구(input)**를 가지고 있습니다.

### 2-4. 연산자(Op) 목록 확인

In [ ]:
# [분석] 모델이 어떤 연산들로 구성되어 있는지 확인합니다.
# 만약 'Conv', 'Relu' 외에 듣도 보도 못한 연산자가 있다면 특정 하드웨어에서 안 돌아갈 수도 있습니다.
op_types = set()
for node in onnx_model.graph.node:
    op_types.add(node.op_type)

print("사용된 ONNX 연산자 목록:")
print("=" * 30)
for op in sorted(op_types):
    print(f"  - {op}")

사용된 ONNX 연산자 목록:
  - Conv
  - Gemm
  - MaxPool
  - Relu
  - Reshape


# 모델을 구성하는 핵심 연산자(부품) 목록

이 리스트는 모델이라는 기계를 구성하고 있는 **'핵심 부품(연산자)들의 목록'**입니다. 각 부품이 모델 내부에서 어떤 역할을 하는지 알기 쉽게 설명해 드릴게요.

---

## 1. Conv (Convolution)
- **역할:** 이미지에서 특징(선, 면, 질감 등)을 찾아내는 **돋보기**  
- **매칭:** PyTorch의 `nn.Conv2d`  
- **설명:** 필터를 이용해 이미지의 구석구석을 훑으며 중요한 특징 데이터를 추출합니다.

---

## 2. Gemm (General Matrix Multiplication)
- **역할:** 복잡한 숫자 행렬을 한꺼번에 곱하고 더하는 **핵심 계산기**  
- **매칭:** PyTorch의 `nn.Linear` (Fully Connected Layer)  
- **설명:** 추출된 특징들을 종합해서 "이 사진은 7일 확률이 높다"라고 최종 판단을 내리는 두뇌 역할을 합니다.

---

## 3. MaxPool (Maximum Pooling)
- **역할:** 중요한 정보만 남기고 데이터 크기를 줄이는 **요약기**  
- **매칭:** PyTorch의 `nn.MaxPool2d`  
- **설명:** 정해진 구역에서 가장 큰 값(가장 강한 특징)만 뽑아내서 다음 레이어로 넘깁니다. 데이터가 압축되어 계산 효율이 좋아집니다.

---

## 4. Relu (Rectified Linear Unit)
- **역할:** 의미 없는 마이너스(-) 데이터는 0으로 걸러내는 **필터**  
- **매칭:** PyTorch의 `nn.ReLU`  
- **설명:** 0보다 작은 값은 죽이고, 0보다 큰 값만 그대로 통과시켜 신경망에 **비선형성**이라는 생동감을 불어넣습니다.

---

## 5. Reshape
- **역할:** 데이터의 형태를 용도에 맞게 바꾸는 **변신 도구**  
- **매칭:** PyTorch의 `x.view()` 또는 `torch.reshape()`  
- **설명:** 예시 코드에서 3차원인 이미지 데이터를 1차원 줄로 길게 펼쳐서(Flatten) Gemm(Linear) 레이어에 넣기 위해 사용되었습니다.

---

## 💡 왜 이 목록을 확인하는 게 중요할까요?
노트북 주석에도 적혀 있듯이, **부품 호환성** 때문입니다.  

삼성/애플 스마트폰의 AI 칩(NPU)이나 특정 임베디드 장비는 **표준적인 연산자(Conv, Relu 등)**는 아주 잘 돌리지만, 최신 논문에서 갓 나온 특이한 연산자는 처리하지 못하는 경우가 많습니다.  

위 리스트에 나온 연산자들은 아주 표준적이고 대중적인 것들이라,  
👉 **"이 모델은 웬만한 기기 어디서든 문제없이 잘 돌아가겠구나!"** 라고 안심하고 배포해도 된다는 뜻입니다.

### 🎯 실습 2-1: PyTorch와 ONNX 출력 비교

같은 입력에 대해 PyTorch 모델과 ONNX 모델의 출력이 동일한지 확인해보기

# ONNX 변환 검증 과정의 의미

이 과정을 수행하는 이유는 한마디로 **"번역이 잘 되었는지 최종 확인하는 검수 작업"**입니다.  
한국어 소설을 영어로 번역했을 때, 줄거리가 바뀌지 않았는지 원본과 대조해 보는 것과 같은 원리입니다.

---

## 1. 번역 오류 적발 (Correctness)
- PyTorch 코드를 ONNX라는 표준 설계도로 바꾸는 과정은 매우 복잡합니다.  
- 드문 경우지만 특정 레이어의 연산이 변환 과정에서 꼬이거나 데이터 규격(Shape)이 잘못 맞춰질 수 있습니다.  
- 예: PyTorch는 `0.9`를 출력하는데 ONNX는 `0.2`를 출력한다면 → 변환 실패, 모델 사용 불가.

---

## 2. 허용 가능한 오차 확인 (Tolerance)
- 컴퓨터는 소수점 계산 시 연산 순서나 하드웨어 방식에 따라 아주 미세한 차이를 냅니다. (예: `0.0000001`)  
- 우리는 출력값을 대조하면서  
  **"오차가 0은 아니지만, 소수점 5째 자리까지는 똑같네? 그럼 안심하고 써도 되겠다."**  
  라는 확신을 얻습니다.

---

## 3. 신뢰도 파악
- 모델을 실제 서비스(아이폰, 웹 등)에 올리기 전에,  
  **"원본 모델과 99.99% 똑같이 동작한다"**라는 데이터 근거가 필요합니다.  
- 이 비교 결과가 바로 그 근거(Report)가 됩니다.

---

## 🍜 비유로 이해하기
- **PyTorch 추론:** "원조 맛집"의 음식 맛  
- **ONNX 추론:** "프랜차이즈 1호점"의 음식 맛  
- **검증 과정:** 1호점 음식을 먹어보고  
  **"원조랑 맛이 똑같네! 드디어 오픈해도 되겠다."**  
  라고 결정하는 최종 맛 테스트.

---

## ✅ 결론
이 단계를 통과해야만 비로소 **"모델 변환이 성공적으로 끝났다"**라고 선언할 수 있으며,  
그 다음 단계(속도 측정, 배포 등)로 넘어갈 수 있습니다.

In [ ]:
import numpy as np
import torch
import onnxruntime as ort

# ============================================================
# ✅ 목표: PyTorch 모델 출력과 ONNX Runtime 출력이 "거의 같은지" 확인하기
#
# 왜 "완전히 같아야" 하지 않고 "거의 같은지" 보나?
# - FP32/FP16 연산 순서, 최적화(상수 폴딩), 런타임 구현 차이 때문에
#   아주 작은 오차(반올림 오차)가 생길 수 있음
# - 그래서 tolerance(허용 오차)를 두고 비교하는 것이 일반적
# ============================================================

# 1) 테스트 입력 생성 (모델이 받는 입력 형태와 동일해야 함: [batch, channel, H, W])
test_input = torch.randn(1, 1, 28, 28)

# 2) PyTorch 모델 추론
model.eval()  # ✅ Dropout/BatchNorm 같은 레이어가 있으면 eval 모드가 중요
with torch.no_grad():  # ✅ 추론에서는 gradient 계산이 필요 없어서 메모리/속도 절약
    pytorch_output = model(test_input).cpu().numpy()  # numpy로 변환 (비교용)

# 3) ONNX Runtime으로 추론
# - ort.InferenceSession은 ONNX 모델을 로드해서 실행하는 객체
# - 입력은 numpy 배열로 넣어야 함
ort_session = ort.InferenceSession(onnx_path)

# [기계적 구동] ONNX 런타임은 C++로 짜여진 고성능 엔진 위에서 돌아갑니다.
# 파이썬은 그저 입력을 던져주고(input_feed), 결과를 받아오는(output_fetch) 역할만 합니다.
onnx_output = ort_session.run(
    None,  # None이면 "모든 출력"을 반환
    {"input": test_input.cpu().numpy()}  # ✅ export할 때 input_names=['input'] 했기 때문에 키가 'input'
)[0]  # 출력이 여러 개면 리스트로 오므로 첫 번째 출력만 사용

# 4) 출력 일부 확인 (상위 5개 값만 프린트)
print("PyTorch 출력 (상위 5개):")
print(pytorch_output[0][:5])

print("\nONNX 출력 (상위 5개):")
print(onnx_output[0][:5])

# 5) 오차 계산
# max_diff = 두 출력 사이에서 가장 큰 절대 차이 (최악의 케이스)
max_diff = np.abs(pytorch_output - onnx_output).max()
print(f"\n최대 차이(max abs diff): {max_diff:.2e}")

# ------------------------------------------------------------
# ✅ 동일 여부(allclose)란?
# np.allclose(A, B, atol=..., rtol=...) 는
# "A와 B가 허용 오차 범위 안에서 거의 같은가?"를 True/False로 알려줌
#
# - atol(absolute tolerance): 절대 오차 허용치 (여기서는 1e-5)
#   예: |A - B| <= 1e-5 이면 '같다'고 인정
#
# - rtol(relative tolerance): 상대 오차 허용치 (기본값도 존재)
#   값의 크기가 클 때는 상대 오차 기준도 함께 고려됨
#
# ✅ 왜 tolerance가 필요?
# - ONNX 변환/런타임 최적화 때문에 아주 미세한 반올림 차이가 생길 수 있음
# - 그래서 "완벽히 동일" 대신 "실용적으로 동일"을 판단함
# ------------------------------------------------------------
is_same = np.allclose(pytorch_output, onnx_output, atol=1e-5)
print(f"동일 여부 (허용 오차 atol=1e-5): {is_same}")

PyTorch 출력 (상위 5개):
[-0.20871101  0.10812928 -0.19496955  0.05748934  0.04447991]

ONNX 출력 (상위 5개):
[-0.20871104  0.10812925 -0.19496953  0.05748926  0.04447993]

최대 차이(max abs diff): 7.82e-08
동일 여부 (허용 오차 atol=1e-5): True


---
## 실습 3: ONNX Runtime으로 추론하기

### 학습
- ONNX Runtime 사용법 익히기
- 추론 속도 비교
- 배치 처리 실습

### 3-1. ONNX Runtime 세션 생성

In [ ]:
import onnxruntime as ort

# ======================================================================================
# ✅ 1단계: 사용 가능한 하드웨어 가속 팀(Execution Provider, EP) 확인
# --------------------------------------------------------------------------------------
# - '프로바이더(EP)'란? : 모델의 복잡한 수치 계산을 어떤 부품(CPU, GPU 등)에 맡길지 결정하는 전담 부서입니다.
# - [코드] ort.get_available_providers()
# - [진짜 의미] "내 PC에 설치된 ONNX 런타임아, 현재 너랑 협력할 수 있는 하드웨어 팀(CPU, GPU 등) 리스트 좀 줘봐."
# - 결과 예시: ['AzureExecutionProvider', 'CPUExecutionProvider']
#   -> "주인님, 지금은 클라우드 팀이랑 일반 CPU 팀만 출근해 있네요."라고 대답하는 것입니다.
# - 결과 분석:
#   1) AzureEP: 현재 실행 환경(Azure 클라우드 등)에서 지원하는 특수 가속 지원팀
#   2) CPUEP: 모든 컴퓨터에 기본으로 있는 부서, 가속기가 없어도 돌아가게 해주는 최후의 보험
# ======================================================================================
print("사용 가능한 Execution Providers:")
print(ort.get_available_providers())


# ======================================================================================
# ✅ 2단계: 엔진 최적화 옵션 설정 (SessionOptions)
# --------------------------------------------------------------------------------------
# - [코드] session_options = ort.SessionOptions()
# - [진짜 의미] "이제부터 모델을 돌릴 건데, 그냥 돌리지 말고 특별 옵션을 좀 넣고 싶어.
#                 옵션을 적어둘 빈 양식(객체)을 하나 만들어줘."
# - ORT_ENABLE_ALL: ONNX 런타임의 모든 최적화 기능을 풀가동합니다.
#   1) 프루닝(Pruning): 최종 결과에 영향이 없는 '죽은 노드(계산)'들을 지도에서 아예 파버립니다.
#   2) 퓨전(Fusion): "A계산 하고 메모리에 썼다가 다시 읽어서 B계산 하기" 대신,
#      "A와 B를 한꺼번에 계산하기"로 공정을 합쳐서 속도를 획기적으로 높입니다. (예: Conv + ReLU 합치기)
# ======================================================================================
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL


# ======================================================================================
# ✅ 3단계: 추론 세션(InferenceSession) 생성
# --------------------------------------------------------------------------------------
# - [코드] ort_session = ort.InferenceSession(...)
# - [진짜 의미] "자, 이제 진짜 일을 시작한다. 다음 3가지를 들고 가서 작업대(Session)를 차려!"
#    1. onnx_path: 아까 만든 모델 지도 파일
#    2. sess_options: 방금 적은 '다이어트(최적화)' 옵션지
#    3. providers: 일을 맡길 부서 (여기서는 'CPU 부서'한테 시키겠다!)
# - 이 코드가 실행되는 순간: 모델 파일을 읽고, 위에서 설정한 지도의 최적화(합치기/쳐내기)를 실제로 수행합니다.
# - providers=['CPUExecutionProvider']: 계산 업무를 맡길 부서의 우선순위를 정합니다.
#   (만약 GPU용 CUDA 팀을 쓰고 싶다면 ['CUDAExecutionProvider', 'CPUExecutionProvider'] 순으로 적습니다.)
# - [결과] 이 줄이 끝나면 'ort_session'이라는 완벽한 계산 엔진(작업대)이 탄생합니다.
# ======================================================================================
ort_session = ort.InferenceSession(
    onnx_path,
    sess_options=session_options,
    providers=['CPUExecutionProvider'] 
)

print("\nONNX Runtime 세션 생성 완료!")

사용 가능한 Execution Providers:
['AzureExecutionProvider', 'CPUExecutionProvider']

ONNX Runtime 세션 생성 완료!


### 3-2. 입출력 정보 확인

In [ ]:
# 입력 정보
# [기계적 구동] 모델 파일을 뜯어보지 않아도 세션 객체를 통해 입출력 스펙을 알 수 있습니다.
print("입력 정보:")
for input_meta in ort_session.get_inputs():
    print(f"  이름: {input_meta.name}")
    print(f"  Shape: {input_meta.shape}")
    print(f"  Type: {input_meta.type}")

print()

# 출력 정보
print("출력 정보:")
for output_meta in ort_session.get_outputs():
    print(f"  이름: {output_meta.name}")
    print(f"  Shape: {output_meta.shape}")
    print(f"  Type: {output_meta.type}")

입력 정보:
  이름: input
  Shape: ['s77', 1, 28, 28]
  Type: tensor(float)

출력 정보:
  이름: output
  Shape: ['s77', 10]
  Type: tensor(float)


### 3-3. 추론 속도 비교: PyTorch vs ONNX Runtime

In [ ]:
import time

# --------------------------------------------------------------------------------------
# 1단계: 시합 규칙 정하기
# --------------------------------------------------------------------------------------
batch_size = 32      # 한 번에 32장의 이미지를 동시에 처리한다!
num_iterations = 100 # 이 짓을 100번 반복해서 평균을 내겠다! (정확도 확보)

# --------------------------------------------------------------------------------------
# 2단계: 선수별 맞춤 데이터 준비
# --------------------------------------------------------------------------------------
# [테스트 데이터] 32장, 1채널, 28x28 크기의 가짜 이미지 생성
test_input_torch = torch.randn(batch_size, 1, 28, 28)
# [선수1용] PyTorch는 전용 'Tensor' 형식을 좋아함
test_input_numpy = test_input_torch.numpy()
# [선수2용] ONNX Runtime은 범용적인 'Numpy' 형식을 좋아함

# --------------------------------------------------------------------------------------
# 3단계: 선수 1 - PyTorch 측정 (원조 맛집)
# --------------------------------------------------------------------------------------
model.eval()         # "공부(Train) 끝났으니 이제 실전(Eval)이다!"라고 선언
with torch.no_grad():# "정답 맞힐 생각만 하고, 공부 흔적(Gradient)은 남기지 마!" (메모리 절약)
    
    # [Warm-up] 아주 중요! 
    # 기계도 시동을 걸자마자 최고 속도가 안 나옵니다. 
    # 처음 10번은 '공회전'을 시켜서 CPU 캐시 등을 미리 달궈놓는 과정입니다.
    for _ in range(10):
        _ = model(test_input_torch)

    # 진짜 시합 시작
    start = time.time() # 시계 시작!
    for _ in range(num_iterations):
        _ = model(test_input_torch)
    # [계산] (걸린 시간 / 100번) * 1000 -> 1회당 몇 ms(0.001초) 걸렸나?
    pytorch_time = (time.time() - start) / num_iterations * 1000

# --------------------------------------------------------------------------------------
# 4단계: 선수 2 - ONNX Runtime 측정 (튜닝 엔진)
# --------------------------------------------------------------------------------------
# [Warm-up] 마찬가지로 10번 공회전
for _ in range(10):
    _ = ort_session.run(None, {'input': test_input_numpy})

# 진짜 시합 시작
start = time.time()
for _ in range(num_iterations):
    # ort_session.run: "아까 차려놓은 작업대(Session)에서 계산 돌려!"
    _ = ort_session.run(None, {'input': test_input_numpy})
onnx_time = (time.time() - start) / num_iterations * 1000

추론 속도 비교 (배치 크기: 32, 반복: 100회)
PyTorch:      12.287 ms/batch
ONNX Runtime: 3.550 ms/batch

속도 향상: 3.46x


# 📊 결과 해석 (PyTorch vs ONNX Runtime)

## 1. PyTorch (12.287 ms)
- **환경:** 파이썬 위에서 실행  
- **특징:** 파이썬은 유연하지만, 줄 단위로 코드를 해석해야 하는 **인터프리터 오버헤드**가 존재합니다.  
- **결과:** 아주 미세하게 시간이 더 걸립니다.

---

## 2. ONNX Runtime (3.550 ms)
- **환경:** C++로 작성된 고성능 엔진  
- **특징:** 파이썬은 단순히 "계산해!"라고 명령만 내리고, 실제 계산은 **최적화된 C++ 엔진**이 직접 하드웨어를 제어합니다.  
- **결과:** 훨씬 빠른 경로로 계산이 진행되어 속도가 크게 향상됩니다.

---

## 3. 속도 향상 (3.46x)
- **의미:** "ONNX로 바꿨더니 약 3.5배나 더 빨라졌다"는 뜻입니다.  
- **핵심:** 단순히 변환만 해도 성능이 크게 개선됩니다.

---

## 💡 교훈
1. **배포는 ONNX**  
   - 연구 단계에서는 PyTorch가 편리하지만,  
   - 실제 서비스(서버 대기 시간, 비용 최적화)에서는 ONNX로 변환만 해도 **서버 비용과 응답 시간을 3배 이상 절약**할 수 있습니다.

2. **배치 처리의 힘**  
   - 이미지를 한 장씩 100번 처리하는 것보다,  
   - 32장씩 묶어서 처리하는 것이 하드웨어 효율을 훨씬 높입니다.  
   - 병렬성과 메모리 활용 최적화 덕분에 처리 속도가 크게 개선됩니다.

### 🎯 실습 3-1: 배치 크기별 처리량 비교

다양한 배치 크기에서 ONNX Runtime의 처리량(throughput)을 측정해보기

In [ ]:
# --------------------------------------------------------------------------------------
# 1단계: 실험할 시나리오 목록 만들기
# --------------------------------------------------------------------------------------
# [코드] batch_sizes = [1, 4, 16, 32, 64, 128]
# [진짜 의미] "한 번에 1개씩 처리할 때부터 128개씩 묶어서 처리할 때까지, 총 6가지 상황을 테스트해 볼 거야."
# [비유] "택배 상자를 1개씩 옮길 때랑 128개씩 트럭에 실어 보낼 때 중 언제 제일 많이 보내는지 확인해 보자!"
batch_sizes = [1, 4, 16, 32, 64, 128]
num_iterations = 50 # 정확도를 위해 각 상황마다 50번씩 반복 실험!

# --------------------------------------------------------------------------------------
# 2단계: 각 배치 크기별로 루프(반복문) 돌리기
# --------------------------------------------------------------------------------------
for batch_size in batch_sizes:
    
    # [코드] test_input = np.random.randn(batch_size, 1, 28, 28).astype(np.float32)
    # [진짜 의미] "현재 설정된 배치 크기(예: 32)에 맞는 가짜 이미지를 생성해. 
    #             그리고 ONNX가 좋아하는 실수(float32) 형식으로 딱 맞춰놔."
    test_input = np.random.randn(batch_size, 1, 28, 28).astype(np.float32)

    # ----------------------------------------------------------------------------------
    # 3단계: 엔진 예열 (Warm-up)
    # ----------------------------------------------------------------------------------
    # [코드] for _ in range(5): _ = ort_session.run(...)
    # [진짜 의미] "본 게임 시작 전에 5번만 미리 돌려봐. 엔진에 기름칠(캐시 로딩) 좀 해놔야 측정이 정확하거든."
    for _ in range(5):
        _ = ort_session.run(None, {'input': test_input})

    # ----------------------------------------------------------------------------------
    # 4단계: 진짜 시합 및 시간 측정
    # ----------------------------------------------------------------------------------
    # [코드] start = time.time()
    # [진짜 의미] "자, 지금부터 스톱워치 누른다! 시작!"
    start = time.time()
    for _ in range(num_iterations):
        _ = ort_session.run(None, {'input': test_input}) # 50번 연속 계산!
    elapsed = time.time() - start # "스톱워치 멈춰! 총 얼마나 걸렸어?"

    # ----------------------------------------------------------------------------------
    # 5단계: 성적표(지표) 계산하기
    # ----------------------------------------------------------------------------------
    
    # [지표 1] Latency (지연 시간/응답 속도)
    # [공식] (총 걸린 시간 / 50번) * 1000
    # [진짜 의미] "한 묶음(Batch)을 처리하는 데 평균 몇 밀리초(ms) 걸렸어? (짧을수록 빠름)"
    latency_ms = (elapsed / num_iterations) * 1000
    
    # [지표 2] Throughput (처리량/가성비)
    # [공식] (묶음 크기 * 50번) / 총 걸린 시간
    # [진짜 의미] "그래서 결국 1초당 총 몇 장의 이미지를 처리할 수 있는 거야? (클수록 가성비 좋음)"
    throughput = (batch_size * num_iterations) / elapsed

    # 성적표 한 줄 출력
    print(f"{batch_size:^12} | {latency_ms:^15.3f} | {throughput:^20.1f}")

배치 크기별 처리량 (ONNX Runtime)
 Batch Size  |  Latency (ms)   | Throughput (samples/s)
------------------------------------------------------------
     1       |      0.209      |        4790.8       
     4       |      0.462      |        8666.0       
     16      |      1.704      |        9391.7       
     32      |      3.749      |        8534.7       
     64      |      6.828      |        9373.8       
    128      |     14.494      |        8831.5       


# 📊 결과표 해설 (비즈니스적 관점)

실행 결과를 보면 아주 흥미로운 현상이 나타납니다.

---

## 결과표

| Batch Size | Latency (ms) | Throughput (samples/s) | 해설 |
|------------|--------------|-------------------------|------|
| **1**      | 0.209        | 4790.8                  | [속도는 빠르지만 비효율] 한 번에 1개씩 처리하니 응답은 빠르지만, 매번 엔진에 명령을 내리는 오버헤드가 커서 전체 처리량(4천대)은 낮습니다. |
| **16**     | 1.704        | 9391.7                  | [가성비 폭발] 한 번에 16개씩 묶어주니 하드웨어가 쉬지 않고 병렬로 일하기 시작합니다. 처리량이 약 9천대로 2배 가까이 뜁니다! |
| **64**     | 6.828        | 9373.8                  | [한계 도달] 64장까지는 처리량이 높게 유지되지만, 16장일 때와 처리량(9300대)이 비슷합니다. 즉, 이미 하드웨어의 최대 성능을 다 쓰고 있다는 뜻입니다. |
| **128**    | 14.494       | 8831.5                  | [역효과 발생] 오히려 처리량이 줄어듭니다. 너무 많은 데이터를 한 번에 넣으니 메모리 부하가 걸리거나 캐시 적중률이 떨어져서 효율이 나빠지기 시작한 것입니다. |

---

## 💡 최종 결론 (이 실험을 왜 하는가?)

만약 당신이 서비스를 운영하는 엔지니어라면, 이 표를 보고 다음과 같은 전략을 세울 수 있습니다.

- **실시간 응답이 제일 중요하다!** (예: 카메라 필터, 실시간 스트리밍)  
  👉 **Batch Size 1~4**를 선택합니다.  
  (Throughput은 조금 손해보더라도 Latency를 최소화)

- **한꺼번에 많은 데이터를 분석해야 한다!** (예: 사진 앨범 자동 분류, 대규모 배치 처리)  
  👉 **Batch Size 16~64**를 선택합니다.  
  (개별 응답은 조금 늦더라도, 1초당 처리하는 전체 장수가 2배 이상 늘어나 서버 비용을 절반으로 절약 가능)

---

## 🎯 핵심 메시지
이 실험은 결국 **"내 모델과 내 하드웨어가 가장 신나게 일할 수 있는 스위트 스팟(Sweet Spot)을 찾는 과정"**이었습니다.

### 3-4. 파일 크기 비교

In [ ]:
# --------------------------------------------------------------------------------------
# 1단계: 비교할 파일들의 목록(대상) 만들기
# --------------------------------------------------------------------------------------
# [코드] files = { '이름': '경로', ... }
# [진짜 의미] "나중에 출력할 때 헷갈리지 않게 '예쁜 이름'과 '실제 파일 주소'를 짝지어서 명단(Dictionary)을 만들어놔."
files = {
    'PyTorch state_dict (.pth)': 'saved_models/model_state_dict.pth',
    'PyTorch 전체 모델 (.pth)': 'saved_models/model_full.pth',
    'ONNX 모델 (.onnx)': 'saved_models/model.onnx',
}

print("모델 파일 크기 비교")
print("=" * 60)

# --------------------------------------------------------------------------------------
# 2단계: 명단을 하나씩 꺼내서 실제 용량 재기
# --------------------------------------------------------------------------------------
for name, path in files.items():
    # [코드] if os.path.exists(path):
    # [진짜 의미] "파일이 실제로 폴더에 있는지 먼저 확인해봐. 없는데 용량 재라고 하면 에러 나니까!"
    if os.path.exists(path):
        
        # [코드] size = os.path.getsize(path)
        # [진짜 의미] "파일의 크기를 '바이트(Byte)' 단위로 읽어와."
        size = os.path.getsize(path)
        
        # [코드] print(f"... | {size/1024:>10.2f} KB")
        # [진짜 의미] "바이트는 숫자가 너무 크니까 1024로 나눠서 '킬로바이트(KB)'로 바꿔서 보여줘. 소수점 둘째 자리까지 예쁘게 정렬해서!"
        print(f"{name:35} | {size/1024:>10.2f} KB")

모델 파일 크기 비교
PyTorch state_dict (.pth)           |    1650.64 KB
PyTorch 전체 모델 (.pth)                |    1652.49 KB
ONNX 모델 (.onnx)                     |      11.75 KB


# 📊 결과 해석 (왜 이렇게 큰 차이가 날까요?)

결과표를 보면 놀라운 차이가 있습니다.

---

## 결과표

| 모델 방식       | 파일 크기   | 해설 |
|-----------------|-------------|------|
| **PyTorch (.pth)** | 약 1650 KB | [알맹이 가득] 모델의 파라미터(약 42만 개)를 32비트 실수(FP32)로 저장한 원래 크기입니다. (42만 × 4바이트 ≒ 1.6MB) |
| **ONNX (.onnx)**   | 11.75 KB   | [압도적 가벼움] 약 140배나 작아졌습니다. |

---

## ❓ 왜 ONNX가 이렇게 작나요? (매우 중요!)

단순히 "파일 포맷이 좋아서"라고 하기엔 너무나 큰 차이입니다. 여기에는 몇 가지 핵심 요인이 있습니다.

1. **그래프 최적화 (Constant Folding)**  
   - ONNX 변환 시 `do_constant_folding=True` 옵션을 사용했습니다.  
   - 모델 내부에서 미리 계산 가능한 부분들을 실제 값으로 치환하여 저장 → 용량 대폭 감소.

2. **데이터 공유 및 압축**  
   - ONNX는 모델 내부의 중복된 구조를 효율적으로 표현합니다.  
   - 동일한 연산이나 패턴을 중복 저장하지 않고 참조 방식으로 관리.

3. **외부 가중치 저장 (가능성)**  
   - 경우에 따라 ONNX는 모델의 **구조(뼈대)**만 저장하고, **가중치(살)**는 별도로 처리하기도 합니다.  
   - 여기서는 11KB라는 수치로 보아, 모델의 복잡한 연결 고리를 **효율적인 Protobuf 포맷**으로 압축했음을 의미합니다.

---

## 💡 결론

이 실험의 목적은 단순히 **속도 향상**뿐 아니라,  
**용량 측면에서도 ONNX가 배포에 얼마나 유리한가**를 확인하는 것이었습니다.  

특히 **모바일 앱이나 임베디드 장치**처럼 저장 공간이 제한된 환경에서는  
- **1.6MB vs 11KB**의 차이가 엄청난 메리트가 됩니다.  

---

## 🎯 최종 메시지

이제 모델을 만들고 (**PyTorch**),  
번역하고 (**ONNX**),  
검증하고 (**Checker**),  
속도와 용량까지 측정했으니…  

👉 **모델 배포의 전 과정을 완벽히 마스터하신 셈입니다! 🥳**

---
## 📝 Part 2 정리

### 핵심 개념
1. **PyTorch 저장 방식**
   - `state_dict`: 가중치만 저장 (권장)
   - 전체 모델 저장: 코드 의존성 있음

2. **ONNX (Open Neural Network Exchange)**
   - 프레임워크 독립적인 표준 포맷
   - 다양한 추론 엔진에서 사용 가능
   - `torch.onnx.export()`로 변환

3. **ONNX Runtime**
   - 고성능 크로스플랫폼 추론 엔진
   - CPU, GPU, NPU 등 다양한 하드웨어 지원
   - 자동 그래프 최적화

### 변환 파이프라인
```
PyTorch 모델 → ONNX → ONNX Runtime (추론)
                  ↘ TensorRT (NVIDIA GPU 최적화)
                  ↘ CoreML (Apple 디바이스)
                  ↘ OpenVINO (Intel 최적화)
```


### 📊 Part 2 모델 포맷과 ONNX 실습: 결과 해석 및 분석

이 문서는 `onnx_conversion_lab.py` 코드를 통해 얻은 결과가 무엇을 의미하는지,  
특히 **"PyTorch 놔두고 왜 굳이 ONNX로 바꾸나?"** 에 대한 해답을 제시합니다.

---

### 1. 💾 PyTorch 모델 저장 방식 비교 (실습 1 결과)

| 저장 방식        | 저장된 내용                  | 파일 크기 | 안전성 | 비고 |
|------------------|------------------------------|-----------|--------|------|
| state_dict 저장  | 오직 **숫자(가중치)**만 저장 | 작음      | 높음 ✅ | 클래스 코드(`class SimpleCNN...`)가 반드시 별도로 필요함 |
| 전체 모델 저장   | 숫자 + 클래스 코드 + 경로 정보 | 큼        | 낮음 ❌ | 파일 경로가 바뀌거나 라이브러리 버전이 다르면 로딩 실패함 |

💡 **왜 state_dict를 써야 하나요?**  
- **유연성:** `model_full.pth`는 저장할 때 사용했던 파이썬 파일 경로(`project/models.py`)까지 기억합니다.  
  → 폴더 구조를 조금만 바꿔도 `ModuleNotFoundError`를 뿜으며 로딩이 안 됩니다.  
- **현업 표준:** 현업에서는 가중치(state_dict)만 파일 서버에 올리고, 모델 구조 코드는 Git으로 관리하여 버전을 맞추는 방식을 씁니다.

---

### 2. 🔄 ONNX 변환과 정밀도 검증 (실습 2 결과)

❓ **최대 오차(Max Difference)가 0이 아닌 이유**  
- 실습 결과 `max_diff`가 0.00000000이 아니라 `1.43e-07` 같은 아주 작은 숫자가 나왔을 겁니다.  

**부동소수점 연산 순서 차이:**  
- PyTorch: `(A + B) + C` 순서로 계산할 수 있음  
- ONNX Runtime: 최적화를 위해 `A + (B + C)` 순서로 계산할 수 있음  

실수는 결합 법칙이 성립하지 않아서(실습 1편 참고), 미세한 오차가 발생합니다.  

**허용 오차(Tolerance):**  
그래서 `np.allclose(atol=1e-5)` 처럼 **"소수점 5째 자리까지 같으면 같은 걸로 치자"** 라고 합의를 봅니다.

---

### 3. 🚀 추론 속도 비교 (실습 3 결과)

**ONNX Runtime(ORT)**이 PyTorch보다 보통 **1.5배 ~ 3배 더 빠릅니다.**

💡 **왜 ONNX Runtime이 더 빠를까요?**  
- **Graph Fusion (연산 합치기):**  
  PyTorch는 Conv 하고 메모리에 쓰고, Relu 하고 메모리에 씁니다.  
  ORT는 Conv + Relu를 한 번에 처리하는 커널(Fused Kernel)을 써서 메모리 접근을 줄입니다.  

- **Constant Folding (상수 미리 계산):**  
  모델 안에 `2 * 3` 같은 계산이 있으면, 실행할 때 계산하지 않고 미리 `6`으로 저장해둡니다.  

- **정적 메모리 할당:**  
  PyTorch는 실행할 때마다 메모리를 잡았다 놓았다(Dynamic) 하지만,  
  ORT는 처음에 그래프를 분석해서 필요한 메모리를 딱 잡아놓고(Static) 재사용합니다.  

---

### 4. 📈 배치 크기(Batch Size)와 처리량(Throughput)

실습 3-1 결과를 보면 배치 사이즈가 커질수록 **Throughput(초당 처리량)**이 증가하다가 어느 순간 멈춥니다.

- **Batch Size 1:** GPU/CPU가 "일 하나 하고 쉬고, 일 하나 하고 쉬고" 하는 상태라 효율이 나쁩니다.  
- **Batch Size 32~64:** 병렬 처리 능력을 풀가동하여 효율이 극대화됩니다.  
- **Batch Size 128~:** 메모리 대역폭의 한계에 도달하거나 캐시가 넘쳐서 성능 증가폭이 줄어듭니다.  

---

### ✅ 결론

서버에 배포할 때는 **적절한 배치 사이즈(보통 16~64)**로 묶어서 처리하는 것이  
사용자 1명씩 처리하는 것보다 훨씬 효율적입니다.